In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path

dump_dir = Path('.')

X = torch.load(dump_dir / 'feat.pth', map_location='cpu').float()
file_name = torch.load(dump_dir / 'file_name.pth', map_location='cpu')
file_path = torch.load(dump_dir / 'file_path.pth', map_location='cpu')

print(f"Embeddings: {X.shape}, {X.dtype}")
print(f"file_name: {file_name.shape}, {file_name.dtype}")
print(f"file_path: {file_path.shape}, {file_path.dtype}")


In [ ]:
def decode_ascii_row(row):
    codes = row.round().clamp(0, 255).to(torch.int).tolist()
    return "".join(chr(c) for c in codes)

def decode_filename(row):
    return decode_ascii_row(row).strip("\x00").strip()

def decode_path(row):
    return decode_ascii_row(row).rstrip("~").strip("\x00").strip()

N = X.shape[0]
names = [decode_filename(file_name[i]) for i in range(N)]
paths = [decode_path(file_path[i]) for i in range(N)]
rel_paths = [f"{paths[i]}/{names[i]}" for i in range(N)]

print(f"Total: {N} segments")
print("\nFirst 10:")
for i in range(min(10, N)):
    print(f"{i:04d} -> {rel_paths[i]}")


In [ ]:
df = pd.DataFrame({
    'idx': np.arange(N),
    'path': paths,
    'name': names,
    'relpath': rel_paths
})

print(df.head(20))
print(f"\nShape: {df.shape}")


In [ ]:
df = pd.DataFrame({
    'idx': np.arange(N),
    'path': paths,
    'name': names,
    'relpath': rel_paths
})

print(df.head(20))
print(f"\nShape: {df.shape}")

import umap
import matplotlib.pyplot as plt

embeddings = X.numpy()
print(f"Embeddings shape: {embeddings.shape}")

sample_size = min(354958, len(embeddings))
sample_idx = np.random.choice(len(embeddings), sample_size, replace=False)
embeddings_sample = embeddings[sample_idx]

print(f"\nUMAP reduction {embeddings_sample.shape} -> 2D...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42, verbose=True)
embeddings_2d = reducer.fit_transform(embeddings_sample)


In [ ]:
plt.figure(figsize=(16, 12))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=1, alpha=0.5)
plt.title(f'DINO Embeddings UMAP 2D ({sample_size} segments)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.savefig('dino_umap_2d.png', dpi=150)
plt.show()

In [ ]:
torch.save(torch.tensor(embeddings_2d), 'dino_embeddings_2d.pth')
df.to_csv('dino_metadata.csv', index=False)
print("Saved: dino_embeddings_2d.pth, dino_metadata.csv")


In [ ]:
import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

configs = [
    (10, 3, 0.3),
    (15, 5, 0.3),
    (20, 5, 0.2),
    (8, 3, 0.1),
    (12, 4, 0.15),
]

for min_size, min_samp, eps in configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_size,
        min_samples=min_samp,
        cluster_selection_epsilon=eps,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    
    labels = clusterer.fit_predict(embeddings_2d)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    pct_noise = 100 * n_noise / len(labels)
    
    print(f"({min_size:2d}, {min_samp:2d}, {eps:.2f}) -> Clusters: {n_clusters:3d}, Bruit: {pct_noise:5.1f}%")


In [ ]:
import hdbscan

configs = [
    (30, 10, 0.08),
    (30, 8, 0.1),
    (35, 10, 0.1),
    (30, 12, 0.1),
    (40, 10, 0.1),
    (30, 10, 0.12),
]

for min_size, min_samp, eps in configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_size,
        min_samples=min_samp,
        cluster_selection_epsilon=eps,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    
    labels = clusterer.fit_predict(embeddings_2d)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    pct_noise = 100 * n_noise / len(labels)
    
    print(f"({min_size:2d}, {min_samp:2d}, {eps:.2f}) -> Clusters: {n_clusters:3d}, Bruit: {pct_noise:5.1f}%")


In [ ]:
import hdbscan

configs = [
    (30, 10, 0.08),
    (30, 11, 0.08),
    (30, 12, 0.08),
    (32, 10, 0.08),
    (28, 10, 0.08),
    (30, 10, 0.07),
]

for min_size, min_samp, eps in configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_size,
        min_samples=min_samp,
        cluster_selection_epsilon=eps,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    
    labels = clusterer.fit_predict(embeddings_2d)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    pct_noise = 100 * n_noise / len(labels)
    
    print(f"({min_size:2d}, {min_samp:2d}, {eps:.2f}) -> Clusters: {n_clusters:3d}, Bruit: {pct_noise:5.1f}%")


In [ ]:
import hdbscan
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

configs = [
    (30, 10, 0.08),
    (30, 11, 0.08),
    (30, 12, 0.08),
    (32, 10, 0.08),
    (28, 10, 0.08),
    (30, 10, 0.07),
]

results = []

for min_size, min_samp, eps in configs:
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_size,
        min_samples=min_samp,
        cluster_selection_epsilon=eps,
        metric='euclidean',
        cluster_selection_method='leaf'
    )
    
    labels = clusterer.fit_predict(embeddings_2d)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    pct_noise = 100 * n_noise / len(labels)
    
    mask = labels != -1
    eval_idx = np.random.choice(np.where(mask)[0], min(20000, mask.sum()), replace=False)
    
    sil = silhouette_score(embeddings_2d[eval_idx], labels[eval_idx], metric='euclidean')
    db = davies_bouldin_score(embeddings_2d[eval_idx], labels[eval_idx])
    ch = calinski_harabasz_score(embeddings_2d[eval_idx], labels[eval_idx])
    
    results.append({
        'min_size': min_size,
        'min_samples': min_samp,
        'epsilon': eps,
        'n_clusters': n_clusters,
        'pct_noise': pct_noise,
        'silhouette': sil,
        'davies_bouldin': db,
        'calinski': ch,
        'labels': labels
    })
    
    print(f"({min_size}, {min_samp}, {eps:.2f}) -> Clusters: {n_clusters:3d}, Bruit: {pct_noise:5.1f}%, Sil: {sil:.4f}, DB: {db:.4f}, CH: {ch:.2f}")

df_results = pd.DataFrame([{k: v for k, v in r.items() if k != 'labels'} for r in results])
df_results = df_results.sort_values('silhouette', ascending=False)

print(f"\n{'min_size':<10} {'min_samp':<10} {'eps':<10} {'Clusters':<10} {'Bruit%':<10} {'Sil':<10} {'DB':<10} {'CH':<10}")
print("-" * 90)

for _, row in df_results.iterrows():
    marker = "***" if _ == df_results.index[0] else "   "
    print(f"{marker} {int(row['min_size']):<10} {int(row['min_samples']):<10} {row['epsilon']:<10} {int(row['n_clusters']):<10} {row['pct_noise']:<10.1f} {row['silhouette']:<10.4f} {row['davies_bouldin']:<10.4f} {row['calinski']:<10.2f}")

best_idx = df_results.index[0]
best = results[best_idx]
print(f"\nMEILLEUR: ({best['min_size']}, {best['min_samples']}, {best['epsilon']}) - {best['n_clusters']} clusters, {best['pct_noise']:.1f}% bruit, Sil: {best['silhouette']:.4f}")


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    cluster_selection_epsilon=0.07,
    metric='euclidean',
    cluster_selection_method='leaf'
)

labels = clusterer.fit_predict(embeddings_2d)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
pct_noise = 100 * n_noise / len(labels)

print(f"Clusters: {n_clusters}, Bruit: {pct_noise:.1f}%")

cluster_counts = Counter(labels)
cluster_info = {}
for cluster_id, count in cluster_counts.items():
    cluster_info[cluster_id] = {
        'count': count,
        'percentage': 100 * count / len(labels)
    }

df_viz = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'cluster': labels,
})

df_viz['count'] = df_viz['cluster'].map(lambda x: cluster_info[x]['count'])
df_viz['percentage'] = df_viz['cluster'].map(lambda x: cluster_info[x]['percentage'])

df_viz['hover_text'] = df_viz.apply(
    lambda row: f"<b>Cluster {row['cluster']}</b><br>" +
                f"Segments: {row['count']:,}<br>" +
                f"Pourcentage: {row['percentage']:.2f}%<br>" +
                f"Position: ({row['x']:.2f}, {row['y']:.2f})",
    axis=1
)

fig = px.scatter(
    df_viz,
    x='x',
    y='y',
    color='cluster',
    hover_data={'x': False, 'y': False, 'cluster': False, 'hover_text': False, 'count': False, 'percentage': False},
    custom_data=['hover_text'],
    title=f'DINO HDBSCAN - {n_clusters} clusters, Bruit: {pct_noise:.1f}%',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    color_continuous_scale='Viridis',
)

fig.update_traces(
    hovertemplate='%{customdata[0]}<extra></extra>',
    marker=dict(size=3, opacity=0.6, line=dict(width=0))
)

fig.update_layout(
    width=1200,
    height=800,
    template='plotly_white',
    showlegend=False,
    hovermode='closest',
    font=dict(size=12),
    title=dict(font=dict(size=16))
)

fig.show()

fig.write_html('dino_hdbscan_interactive.html')
print("\nSaved: dino_hdbscan_interactive.html")


In [ ]:
mask_no_noise = labels != -1
embeddings_2d_clean = embeddings_2d[mask_no_noise]
labels_clean = labels[mask_no_noise]

df_viz = pd.DataFrame({
    'x': embeddings_2d_clean[:, 0],
    'y': embeddings_2d_clean[:, 1],
    'cluster': labels_clean,
})

cluster_counts = Counter(labels_clean)
cluster_info = {cid: {'count': cnt, 'percentage': 100*cnt/len(labels_clean)} for cid, cnt in cluster_counts.items()}

df_viz['count'] = df_viz['cluster'].map(lambda x: cluster_info[x]['count'])
df_viz['percentage'] = df_viz['cluster'].map(lambda x: cluster_info[x]['percentage'])
df_viz['hover_text'] = df_viz.apply(
    lambda row: f"<b>Cluster {row['cluster']}</b><br>Segments: {row['count']:,}<br>Pourcentage: {row['percentage']:.2f}%",
    axis=1
)

fig = px.scatter(df_viz, x='x', y='y', color='cluster', custom_data=['hover_text'],
                 title=f'DINO HDBSCAN - {n_clusters} clusters (sans bruit)',
                 labels={'x': 'UMAP 1', 'y': 'UMAP 2'}, color_continuous_scale='Viridis')

fig.update_traces(hovertemplate='%{customdata[0]}<extra></extra>', marker=dict(size=3, opacity=0.6))
fig.update_layout(width=1200, height=800, template='plotly_white', showlegend=False, hovermode='closest')
fig.show()


In [ ]:
def get_audio_from_spectrogram_name(spec_name):
    parts = spec_name.replace('.png', '').split('_')
    audio_file = f"{parts[0]}_{parts[1]}_{parts[2]}.flac"
    channel = parts[3]
    segment_idx = int(parts[4])
    start_time = segment_idx * 10
    end_time = start_time + 10
    return audio_file, start_time, end_time, channel

spec_name_example = df.iloc[0]['name']
audio_file, start, end, channel = get_audio_from_spectrogram_name(spec_name_example)
print(f"Example: {spec_name_example}")
print(f"  -> {audio_file} [{start}s-{end}s] {channel}")


In [ ]:
def download_audio_segment(file_name, start_time, end_time, channel):
    parts = file_name.split('_')
    site = parts[0]
    s3_key = f"2019/{site}/Data/{file_name}"
    
    try:
        buf = io.BytesIO()
        s3_client.download_fileobj(BUCKET, s3_key, buf)
        buf.seek(0)
        audio, sr = librosa.load(buf, sr=None, offset=start_time, duration=end_time - start_time, mono=False)
        
        if len(audio.shape) > 1:
            ch_idx = 0 if channel == 'ch0' else 1
            audio = audio[ch_idx]
        
        return audio, sr
    except Exception as e:
        print(f"Error: {e}")
        return None, None


In [ ]:
def play_cluster_examples(cluster_id, n_examples=2):
    cluster_indices_global = np.where(labels == cluster_id)[0]
    
    if len(cluster_indices_global) == 0:
        print(f"Cluster {cluster_id}: No segments")
        return
    
    sample_indices = np.random.choice(cluster_indices_global, size=min(n_examples, len(cluster_indices_global)), replace=False)
    
    print(f"\nCluster {cluster_id} - {len(cluster_indices_global)} segments")
    print("="*60)
    
    for i, idx in enumerate(sample_indices):
        spec_name = df.iloc[idx]['name']
        audio_file, start_time, end_time, channel = get_audio_from_spectrogram_name(spec_name)
        
        print(f"\nExample {i+1}: {spec_name}")
        print(f"  File: {audio_file}, Time: {start_time}s-{end_time}s, Channel: {channel}")
        
        audio, sr = download_audio_segment(audio_file, start_time, end_time, channel)
        if audio is not None:
            display(Audio(audio, rate=sr))
        else:
            print("  Download failed")

for cluster_id in sorted(set(labels_clean))[:20]:
    play_cluster_examples(cluster_id, n_examples=2)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

feat_pca50 = torch.load('feat_pca50.pth')
embeddings_50d = feat_pca50.numpy() if isinstance(feat_pca50, torch.Tensor) else feat_pca50

print(f"Embeddings 50D: {embeddings_50d.shape}")

k_values = list(range(120, 401, 40))
print(f"K values: {k_values}")

results = []

for k in k_values:
    print(f"\nK={k}")
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = kmeans.fit_predict(embeddings_50d)
    
    eval_size = min(20000, len(embeddings_50d))
    eval_idx = np.random.choice(len(embeddings_50d), eval_size, replace=False)
    
    sil = silhouette_score(embeddings_50d[eval_idx], labels[eval_idx], metric='euclidean', sample_size=10000)
    db = davies_bouldin_score(embeddings_50d[eval_idx], labels[eval_idx])
    ch = calinski_harabasz_score(embeddings_50d[eval_idx], labels[eval_idx])
    
    print(f"  Sil: {sil:.4f}, DB: {db:.4f}, CH: {ch:.2f}")
    
    results.append({
        'k': k,
        'silhouette': sil,
        'davies_bouldin': db,
        'calinski': ch,
        'inertia': kmeans.inertia_
    })

df_kmeans = pd.DataFrame(results)
df_kmeans = df_kmeans.sort_values('silhouette', ascending=False)

print(f"\n{'K':<10} {'Silhouette':<12} {'Davies-B':<12} {'Calinski':<12} {'Inertia':<12}")
print("-" * 60)

for _, row in df_kmeans.iterrows():
    marker = "***" if _ == df_kmeans.index[0] else "   "
    print(f"{marker} {int(row['k']):<10} {row['silhouette']:<12.4f} {row['davies_bouldin']:<12.4f} {row['calinski']:<12.2f} {row['inertia']:<12.2f}")

best = df_kmeans.iloc[0]
print(f"\nMEILLEUR: K={int(best['k'])}, Silhouette: {best['silhouette']:.4f}")


In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=120, random_state=42, n_init=10, max_iter=300)
labels_kmeans = kmeans.fit_predict(embeddings_50d)

print(f"K-Means avec K=120")
print(f"Labels shape: {labels_kmeans.shape}")

cluster_counts = Counter(labels_kmeans)
print(f"\nTop 10 clusters:")
for cluster_id, count in cluster_counts.most_common(10):
    print(f"  Cluster {cluster_id}: {count} segments")


In [ ]:
def play_cluster_examples_kmeans(cluster_id, n_examples=2):
    cluster_indices = np.where(labels_kmeans == cluster_id)[0]
    
    if len(cluster_indices) == 0:
        print(f"Cluster {cluster_id}: No segments")
        return
    
    sample_indices = np.random.choice(cluster_indices, size=min(n_examples, len(cluster_indices)), replace=False)
    
    print(f"\nCluster {cluster_id} - {len(cluster_indices)} segments")
    print("="*60)
    
    for i, idx in enumerate(sample_indices):
        spec_name = df.iloc[idx]['name']
        audio_file, start_time, end_time, channel = get_audio_from_spectrogram_name(spec_name)
        
        print(f"\nExample {i+1}: {spec_name}")
        print(f"  File: {audio_file}, Time: {start_time}s-{end_time}s, Channel: {channel}")
        
        audio, sr = download_audio_segment(audio_file, start_time, end_time, channel)
        if audio is not None:
            display(Audio(audio, rate=sr))
        else:
            print("  Download failed")

for cluster_id in range(20):
    play_cluster_examples_kmeans(cluster_id, n_examples=2)


In [ ]:
import umap

mask_sample = np.random.choice(len(embeddings_50d), size=max(50000, len(embeddings_50d)), replace=False)
embeddings_50d_sample = embeddings_50d[mask_sample]
labels_kmeans_sample = labels_kmeans[mask_sample]

print(f"Sample: {embeddings_50d_sample.shape}")
print(f"Labels sample: {labels_kmeans_sample.shape}")

print("\nUMAP supervisé (utilise les labels K-Means)...")
reducer_supervised = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
    verbose=True
)

embeddings_2d_supervised = reducer_supervised.fit_transform(
    embeddings_50d_sample,
    y=labels_kmeans_sample
)

print(f"Embeddings 2D supervised: {embeddings_2d_supervised.shape}")


In [ ]:
import plotly.express as px

cluster_counts = Counter(labels_kmeans_sample)
cluster_info = {cid: {'count': cnt, 'percentage': 100*cnt/len(labels_kmeans_sample)} for cid, cnt in cluster_counts.items()}

df_viz = pd.DataFrame({
    'x': embeddings_2d_supervised[:, 0],
    'y': embeddings_2d_supervised[:, 1],
    'cluster': labels_kmeans_sample,
})

df_viz['count'] = df_viz['cluster'].map(lambda x: cluster_info[x]['count'])
df_viz['percentage'] = df_viz['cluster'].map(lambda x: cluster_info[x]['percentage'])
df_viz['hover_text'] = df_viz.apply(
    lambda row: f"<b>Cluster {row['cluster']}</b><br>Segments: {row['count']:,}<br>Pourcentage: {row['percentage']:.2f}%",
    axis=1
)

fig = px.scatter(
    df_viz,
    x='x',
    y='y',
    color='cluster',
    custom_data=['hover_text'],
    title=f'K-Means (K=120) + UMAP Supervisé - Meilleure séparation',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    color_continuous_scale='Viridis'
)

fig.update_traces(
    hovertemplate='%{customdata[0]}<extra></extra>',
    marker=dict(size=3, opacity=0.6, line=dict(width=0))
)

fig.update_layout(
    width=1200,
    height=800,
    template='plotly_white',
    showlegend=False,
    hovermode='closest'
)

fig.show()
fig.write_html('kmeans_120_umap_supervised.html')
print("Saved: kmeans_120_umap_supervised.html")


In [ ]:
plt.figure(figsize=(20, 16))
scatter = plt.scatter(
    embeddings_2d_supervised[:, 0],
    embeddings_2d_supervised[:, 1],
    c=labels_kmeans_sample,
    cmap='tab20',
    s=2,
    alpha=0.6
)
plt.colorbar(scatter, label='Cluster')
plt.title('K-Means (K=120) + UMAP Supervisé', fontsize=16)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.tight_layout()
plt.savefig('kmeans_120_umap_supervised.png', dpi=150)
plt.show()


In [ ]:
clusters_labels_tsne = torch.load('cluster_labels_tsne.pt')
X_tsne = torch.load('X_tsne.pt')

clusters_labels_tsne = clusters_labels_tsne.numpy() if isinstance(clusters_labels_tsne, torch.Tensor) else clusters_labels_tsne
X_tsne = X_tsne.numpy() if isinstance(X_tsne, torch.Tensor) else X_tsne

print(f"clusters_labels_tsne: {clusters_labels_tsne.shape}")
print(f"X_tsne: {X_tsne.shape}")
print(f"Clusters uniques: {len(set(clusters_labels_tsne))}")



In [ ]:
cluster_counts_tsne = Counter(clusters_labels_tsne)
print(f"\nTop 10 clusters:")
for cluster_id, count in cluster_counts_tsne.most_common(10):
    print(f"  Cluster {cluster_id}: {count} segments")
def play_cluster_examples_tsne(cluster_id, n_examples=2):
    cluster_indices = np.where(clusters_labels_tsne == cluster_id)[0]
    
    if len(cluster_indices) == 0:
        print(f"Cluster {cluster_id}: No segments")
        return
    
    sample_indices = np.random.choice(cluster_indices, size=min(n_examples, len(cluster_indices)), replace=False)
    
    print(f"\nCluster {cluster_id} - {len(cluster_indices)} segments")
    print("="*60)
    
    for i, idx in enumerate(sample_indices):
        spec_name = df.iloc[idx]['name']
        audio_file, start_time, end_time, channel = get_audio_from_spectrogram_name(spec_name)
        
        print(f"\nExample {i+1}: {spec_name}")
        print(f"  File: {audio_file}, Time: {start_time}s-{end_time}s, Channel: {channel}")
        
        audio, sr = download_audio_segment(audio_file, start_time, end_time, channel)
        if audio is not None:
            display(Audio(audio, rate=sr))
        else:
            print("  Download failed")

for cluster_id in sorted(set(clusters_labels_tsne)):
    play_cluster_examples_tsne(cluster_id, n_examples=2)
